# Generación del conjunto de datos base

**Objetivo:** generar el corpus de imágenes y latentes de referencia que se usarán en
los análisis geométrico y semántico.

El notebook está diseñado para ejecutarse una vez por model en sesiones separadas:
cambia `MODEL_KEY` en la celda de configuración y ejecuta todo.

**Banco de prompts** (`corpus/prompts.json`):
- `base` (20 prompts): diversidad temática → PCA y análisis de distancias
- `attribute` (30 prompts = 15 pares): atributo controlado → direcciones semánticas
  - `color` (5 pares warm/cool)
  - `lighting` (5 pares bright/dark)
  - `style` (5 pares photorealistic/artistic)

Con 3 semillas → 150 muestras por modelo.

---
## 0 — Entorno

In [ ]:
from google.colab import drive
import os, sys, torch

drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = "/content/drive/MyDrive/TFM/"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

os.environ["HF_HOME"] = "/content/.cache/huggingface"

assert torch.cuda.is_available(), "GPU no disponible. Activa T4 en Entorno de ejecución."
print(f"GPU: {torch.cuda.get_device_name(0)}  —  "
      f"VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB")
print(f"HF_HOME  : {os.environ['HF_HOME']}")
print(f"Proyecto : {PROJECT_ROOT}")

In [ ]:
#
req = os.path.join(PROJECT_ROOT, "requirements_colab.txt")
!pip install -q -r {req}
import diffusers, transformers
print(f"diffusers={diffusers.__version__}  transformers={transformers.__version__}")

In [ ]:
import os
from huggingface_hub import login

HF_TOKEN = ""  # <-- pega aquí tu token: "hf_..."

if not HF_TOKEN:
    raise ValueError(
        "HF_TOKEN vacío. Rellena con tu token de "
        "https://huggingface.co/settings/tokens (tipo Read) "
        "y vuelve a ejecutar esta celda."
    )

login(token=HF_TOKEN, add_to_git_credential=False)
os.environ["HF_TOKEN"] = HF_TOKEN
print("Autenticado en Hugging Face.")

---
## 1 — Configuración

In [ ]:
# ── ÚNICO PARÁMETRO A CAMBIAR ENTRE SESIONES ────────────────────────────────
MODEL_KEY   = "sd15"   # "sd15" | "sd21" | "sdxl"
SAVE_IMAGES = True     # False para ahorrar espacio en Drive
# ─────────────────────────────────────────────────────────────────────────────

SEEDS   = [0, 1, 2]

OUT_DIR = os.path.join(PROJECT_ROOT, "data", "latents", MODEL_KEY)
IMG_DIR = os.path.join(OUT_DIR, "images")
os.makedirs(IMG_DIR, exist_ok=True)

print(f"Modelo : {MODEL_KEY}")
print(f"Salida : {OUT_DIR}")

---
## 2 — Banco de prompts

In [ ]:
# Carga del banco de prompts desde data/prompts.json y expansión a la lista plana
# de muestras (prompt × semilla). El banco contiene dos categorías:
#   - "base" (20 prompts): diversidad temática para análisis geométrico y PCA base.
#   - "attribute" (30 prompts = 15 pares controlados): cada par describe la misma
#     escena variando únicamente un atributo (color, iluminación o estilo), lo que
#     permite construir direcciones semánticas puras mediante diferencia de centroides.
# Las 3 semillas por prompt generan 150 muestras totales; promediar sobre semillas
# al construir las direcciones elimina la variabilidad estocástica del denoising.
import json

with open(os.path.join(PROJECT_ROOT, "data", "prompts.json")) as f:
    prompt_bank = json.load(f)

# Expandir a lista plana (prompt × seed)
samples = []
for p in prompt_bank["prompts"]:
    for seed in SEEDS:
        samples.append({**p, "seed": seed})

print(f"Total muestras a generar : {len(samples)}")
cats = {}
for s in samples:
    cats[s["category"]] = cats.get(s["category"], 0) + 1
for k, v in cats.items():
    print(f"  {k:<12} {v} muestras")

---
## 3 — Carga del modelo

In [ ]:
from models import load_model

model = load_model(MODEL_KEY)
print(model)

---
## 4 — Generación de latentes

El bucle guarda un checkpoint cada 25 muestras en `_checkpoint.pt`.
Si la sesión se interrumpe, re-ejecuta esta celda: retoma desde donde lo dejó.

In [ ]:
# Bucle principal de generación con sistema de checkpoint para tolerancia a fallos.
# Colab puede interrumpir la sesión (timeout, desconexión) durante los ~40 min
# que tarda generar 150 muestras. El checkpoint serializa el estado cada 25
# muestras, permitiendo reanudar exactamente donde se interrumpió sin regenerar
# nada. Al terminar, el archivo de checkpoint se elimina.
# El índice "idx" de cada entrada del manifest corresponde a la posición en el
# tensor apilado final, garantizando alineación entre tensor y manifest.
CKPT_PATH        = os.path.join(OUT_DIR, "_checkpoint.pt")
CHECKPOINT_EVERY = 25

# Cargar progreso previo (si existe)
if os.path.exists(CKPT_PATH):
    ckpt        = torch.load(CKPT_PATH, map_location="cpu")
    manifest    = ckpt["manifest"]
    latents_all = ckpt["latents"]
    done        = {(m["prompt"], m["seed"]) for m in manifest}
    print(f"Reanudando: {len(manifest)} muestras ya guardadas.")
else:
    manifest    = []
    latents_all = []
    done        = set()

pending = [s for s in samples if (s["prompt"], s["seed"]) not in done]
print(f"Pendientes: {len(pending)}/{len(samples)}")

for i, sample in enumerate(pending):
    image, z0 = model.generate(sample["prompt"], seed=sample["seed"])

    idx   = len(manifest)
    entry = {**sample, "idx": idx}
    manifest.append(entry)
    latents_all.append(z0.cpu())

    if SAVE_IMAGES:
        image.save(os.path.join(IMG_DIR, f"{idx:05d}.png"))

    # Checkpoint periódico y al final
    if (i + 1) % CHECKPOINT_EVERY == 0 or (i + 1) == len(pending):
        torch.save({"manifest": manifest, "latents": latents_all}, CKPT_PATH)
        print(f"  [{i+1:>3}/{len(pending)}] checkpoint — {len(manifest)} muestras totales")

print(f"\nGeneración completada: {len(manifest)} muestras.")

---
## 5 — Verificación de reproducibilidad

Regenera las primeras 3 muestras y comprueba que el latente es idéntico.

In [ ]:
# Verificación de reproducibilidad: regenera las primeras 3 muestras y comprueba
# que los latentes obtenidos son idénticos bit a bit a los guardados.
# La reproducibilidad es un requisito fundamental del protocolo experimental:
# garantiza que los análisis de las Fases 3 y 4 son comparables entre sesiones
# y entre los tres modelos, ya que todos usan las mismas semillas y prompts.
print("Verificando reproducibilidad (3 muestras)...")
ok_count = 0
for sample in samples[:3]:
    _, z0_new = model.generate(sample["prompt"], seed=sample["seed"])
    # Recuperar latente guardado
    entry     = next(m for m in manifest
                     if m["prompt"] == sample["prompt"] and m["seed"] == sample["seed"])
    z0_saved  = latents_all[entry["idx"]]
    max_diff  = (z0_new.cpu() - z0_saved).abs().max().item()
    status    = "OK" if max_diff < 1e-4 else "FAIL"
    ok_count += (max_diff < 1e-4)
    print(f"  [{status}] seed={sample['seed']}  max_diff={max_diff:.2e}  "
          f"prompt='{sample['prompt'][:50]}'")

print(f"\n{ok_count}/3 muestras reproducibles.")

---
## 6 — Guardar resultados finales

In [ ]:
MANIFEST_PATH = os.path.join(OUT_DIR, "manifest.json")
LATENTS_PATH  = os.path.join(OUT_DIR, "latents.pt")

# Tensor apilado (N, C, H, W)
latents_tensor = torch.stack(latents_all)

with open(MANIFEST_PATH, "w") as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

torch.save(latents_tensor, LATENTS_PATH)

# Eliminar checkpoint temporal
if os.path.exists(CKPT_PATH):
    os.remove(CKPT_PATH)

size_mb = os.path.getsize(LATENTS_PATH) / 1024**2
print(f"latents.pt  : {latents_tensor.shape}  dtype={latents_tensor.dtype}  {size_mb:.1f} MB")
print(f"manifest.json: {len(manifest)} entradas")
print(f"Ruta: {OUT_DIR}")

---
## 7 — Estadísticas del espacio latente

In [ ]:
z = latents_tensor.float()
norms = z.view(len(z), -1).norm(dim=1)

print(f"{'Shape':<22} {str(tuple(z.shape))}")
print(f"{'Dim latente total':<22} {z.shape[1]*z.shape[2]*z.shape[3]:,}")
print(f"{'Media global':<22} {z.mean().item():.4f}")
print(f"{'Std global':<22} {z.std().item():.4f}")
print(f"{'Norma L2 media':<22} {norms.mean().item():.2f}")
print(f"{'Norma L2 std':<22} {norms.std().item():.2f}")

print("\nNorma L2 por categoría:")
print(f"{'Categoría':<22} {'n':>4}  {'media':>8}  {'std':>8}")
print("-" * 46)
for cat in sorted(set(m["category"] for m in manifest)):
    idxs  = [m["idx"] for m in manifest if m["category"] == cat]
    cat_z = z[idxs].view(len(idxs), -1).norm(dim=1)
    print(f"  {cat:<20} {len(idxs):>4}  {cat_z.mean().item():>8.2f}  {cat_z.std().item():>8.2f}")